In [ ]:
!uv add 'markitdown[pdf]'


In [ ]:
import csv
import io
from pathlib import Path
from urllib.parse import unquote, urlparse

import requests

CSV_URL = "https://raw.githubusercontent.com/alexeygrigorev/ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv"

# Support running from the project root or homework/.
project_root = Path.cwd()
if not (project_root / "books").is_dir():
    project_root = project_root.parent
books_dir = project_root / "books"
books_dir.mkdir(parents=True, exist_ok=True)

response = requests.get(CSV_URL, timeout=60)
response.raise_for_status()
csv_text = response.content.decode("utf-8-sig")
rows = list(csv.DictReader(io.StringIO(csv_text)))
if not rows or "pdf_url" not in rows[0]:
    raise ValueError("The CSV must contain book rows and a pdf_url column.")
(books_dir / "books.csv").write_text(csv_text, encoding="utf-8")

failures = []
for row in rows:
    title = row["title"]
    pdf_url = row["pdf_url"].strip()
    filename = Path(unquote(urlparse(pdf_url).path)).name
    pdf_path = books_dir / filename
    try:
        if not filename.lower().endswith(".pdf"):
            raise ValueError(f"Expected a PDF filename in {pdf_url}")
        if pdf_path.exists() and pdf_path.stat().st_size > 0:
            with pdf_path.open("rb") as existing:
                if existing.read(5) == b"%PDF-":
                    print(f"Already downloaded: {filename}")
                    continue
        with requests.get(pdf_url, timeout=120) as pdf_response:
            pdf_response.raise_for_status()
            content = pdf_response.content
        if not content.startswith(b"%PDF-"):
            raise ValueError("The downloaded response is not a PDF.")
        temporary_path = pdf_path.with_suffix(".pdf.part")
        temporary_path.write_bytes(content)
        temporary_path.replace(pdf_path)
        print(f"Downloaded {title} -> {pdf_path.relative_to(project_root)}")
    except Exception as error:
        failures.append(title)
        print(f"Failed to download {title}: {error}")

print(f"Ready: {len(rows) - len(failures)} of {len(rows)} PDFs in {books_dir}")
if failures:
    raise RuntimeError(f"Downloads failed for: {', '.join(failures)}")


In [ ]:
from pathlib import Path
from markitdown import MarkItDown

# Support running the notebook from the project root or homework/.
project_root = Path.cwd()
if not (project_root / "books").is_dir():
    project_root = project_root.parent

pdf_dir = project_root / "books"
output_dir = project_root / "books_text"
output_dir.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(
    path for path in pdf_dir.rglob("*")
    if path.is_file() and path.suffix.lower() == ".pdf"
)

if not pdf_files:
    raise FileNotFoundError(f"No downloaded PDFs found in {pdf_dir}")

converter = MarkItDown()
failures = []
for pdf_path in pdf_files:
    # Preserve any subdirectories to avoid duplicate filename collisions.
    markdown_path = output_dir / pdf_path.relative_to(pdf_dir).with_suffix(".md")
    try:
        result = converter.convert(str(pdf_path))
        markdown_path.parent.mkdir(parents=True, exist_ok=True)
        markdown_path.write_text(result.text_content, encoding="utf-8")
        print(f"Converted {pdf_path.name} -> {markdown_path.relative_to(project_root)}")
    except Exception as error:
        failures.append(pdf_path.name)
        print(f"Failed to convert {pdf_path.name}: {error}")

print(f"Converted {len(pdf_files) - len(failures)} of {len(pdf_files)} PDFs.")
if failures:
    raise RuntimeError(f"Conversion failed for: {', '.join(failures)}")
